Análisis de los documentos disponibles

*Respaldo de los antecedentes del Documento de Negocio (`docs/negocio.md`, sección 1)*

**Punto de inicio.** Tenemos seis cuestionarios recolectados con Google Forms y **ninguna pregunta de investigación definida**. Este cuaderno explora los datos para responder tres preguntas, en este orden:

1. ¿Qué contiene cada cuestionario y quién lo respondió?
2. ¿Qué tema pueden sostener los datos, es decir, qué se mide en más de un cuestionario y para las mismas personas?
3. Una vez elegido el tema, ¿qué cuestionarios sirven para estudiarlo y cuáles no?

Para proteger el anonimato, ninguna celda muestra números de cuenta ni respuestas individuales; solo conteos.

In [1]:
# Configuración: lectura de los 6 documentos
import re
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))
from analisis_documentos import DOCS, NOMBRES, TEMAS, columna, leer, llave, mapa_de_temas, normalizar

pd.set_option("display.max_colwidth", 80)
datos = {n: leer(archivo) for n, (archivo, _) in DOCS.items()}
llaves = {n: llave(datos[n], DOCS[n][1]) for n in DOCS}
print("Documentos leídos:", len(datos))

Documentos leídos: 6


## 1. ¿Qué contiene cada cuestionario?

In [2]:
# Tabla 1. Inventario: tamaño, periodo de respuesta y llave de cada documento
def fechas(df):
    t = (df["Marca temporal"].astype(str).str.replace(r"\s*GMT[+-]\d+", "", regex=True)
         .str.replace(r"a\.\s*m\.", "AM", regex=True).str.replace(r"p\.\s*m\.", "PM", regex=True))
    f = pd.to_datetime(t, errors="coerce", format="mixed")
    return f"{f.min():%d/%m} – {f.max():%d/%m/%Y}"

inventario = pd.DataFrame({
    "documento": NOMBRES,
    "respuestas": {n: len(d) for n, d in datos.items()},
    "preguntas": {n: sum(not re.search(r"marca temporal|correo|numero de cuenta", normalizar(c)) for c in d.columns)
                  for n, d in datos.items()},   # sin contar columnas de fecha ni de identificación
    "periodo": {n: fechas(d) for n, d in datos.items()},
    "llave": {n: "correo" if DOCS[n][1] is None else "número de cuenta" for n in DOCS},
}).rename_axis("doc")
inventario

,documento,respuestas,preguntas,periodo,llave
doc,,,,,
1,Música y estado de ánimo,24,16,29/10 – 11/11/2026,número de cuenta
2,Entorno universitario y bienestar,19,15,01/11 – 03/11/2026,número de cuenta
3,Uso de IA en lo académico,24,16,30/10 – 11/11/2026,número de cuenta
4,Test (personalidad),25,15,04/09 – 11/11/2026,número de cuenta
5,Pelis y series,21,15,30/10 – 04/11/2026,número de cuenta
6,Relación amor–carrera,21,19,12/10 – 13/11/2026,correo


Son seis cuestionarios cortos (entre 15 y 19 preguntas) con 19 a 25 respuestas cada uno, aplicados entre septiembre y noviembre de 2026. Casi todas las preguntas son abiertas: los estudiantes respondieron con sus propias palabras.

## 2. ¿Quién los respondió?

In [3]:
# Tabla 2. Estudiantes en común entre cada par de documentos (por número de cuenta)
conj = {n: set(k.dropna().astype(int)) for n, k in llaves.items()}
comunes = pd.DataFrame([[len(conj[a] & conj[b]) for b in DOCS] for a in DOCS],
                       index=[f"Doc {n}" for n in DOCS], columns=[f"Doc {n}" for n in DOCS])
print("Estudiantes distintos en total:", len(set().union(*conj.values())))
comunes

Estudiantes distintos en total: 26


,Doc 1,Doc 2,Doc 3,Doc 4,Doc 5,Doc 6
Doc 1,24,19,23,23,21,20
Doc 2,19,19,19,19,19,19
Doc 3,23,19,24,24,21,20
Doc 4,23,19,24,24,21,20
Doc 5,21,19,21,21,21,20
Doc 6,20,19,20,20,20,21


In [4]:
# Carrera y semestre (solo los documentos que lo preguntan)
semestre_2 = columna(datos[2], "semestre")
carrera_2 = columna(datos[2], "carrera universitaria")
print("Doc 2 · 7.º semestre:", semestre_2.str.contains(r"7|septimo|setimo").sum(), "de", len(semestre_2))
print("Doc 2 · carrera MAC:", carrera_2.str.contains(r"\bmac\b|matematicas apl").sum(), "de", len(carrera_2))
semestre_6 = datos[6].filter(like="semestre").iloc[:, 0]
print("Doc 6 · 7.º semestre:", int((semestre_6 == 7).sum()), "de", int(semestre_6.notna().sum()))

Doc 2 · 7.º semestre: 19 de 19
Doc 2 · carrera MAC: 18 de 19
Doc 6 · 7.º semestre: 20 de 20


Los seis cuestionarios los respondió **prácticamente el mismo grupo**: estudiantes de 7.º semestre de Matemáticas Aplicadas y Computación. Esto significa que los temas de distintos cuestionarios se pueden **cruzar para las mismas personas**, lo cual es la condición para estudiar relaciones entre ellos.

## 3. ¿De qué temas tratan?

Para ver qué tema podría sostener un estudio, se clasificaron los **enunciados** de todas las preguntas (no las respuestas) en temas, buscando palabras clave en texto normalizado (minúsculas y sin acentos). Una pregunta puede tocar más de un tema.

In [5]:
# Tabla 3. Número de preguntas de cada documento que tocan cada tema
temas = mapa_de_temas(datos)
# Fila combinada: preguntas sobre bienestar emocional (ánimo, estrés o sueño), sin contar dos veces
bienestar = re.compile("|".join(TEMAS[t] for t in ["Ánimo y emociones", "Estrés y carga", "Sueño y descanso"]))
temas.loc["Bienestar emocional (ánimo, estrés o sueño)"] = [
    sum(bool(bienestar.search(normalizar(c))) for c in d.columns[2:]) for d in datos.values()]
temas["documentos que lo tocan"] = (temas > 0).sum(axis=1)
temas

,Doc 1,Doc 2,Doc 3,Doc 4,Doc 5,Doc 6,documentos que lo tocan
Música,16,0,0,0,0,1,2
Ánimo y emociones,7,3,0,0,0,0,2
Estrés y carga,1,0,1,0,0,0,2
Sueño y descanso,1,1,0,0,0,0,2
Concentración y desempeño,1,2,6,0,0,0,3
Inteligencia artificial,0,0,15,0,0,0,1
"Contexto (carrera, semestre, traslado)",0,7,1,0,0,5,3
Cine y series,0,0,0,1,13,1,3
Pareja,0,0,0,0,0,11,1
Personalidad y ocio,0,0,0,8,0,0,1


**Lectura del mapa de temas:**

- **Cine y series, pareja y personalidad** se concentran cada uno en un cuestionario: Pelis y series (13 preguntas), Relación amor–carrera (11) y Test (8). El cine aparece también en el 4 y el 6, pero solo en una pregunta de gustos. Ninguno de los tres temas se conecta con una medida de bienestar.
- **La música** solo está en el Documento 1, pero ahí se mide a fondo (16 preguntas): géneros, horas, momentos, plataformas y su efecto en el ánimo, la energía, la concentración y el sueño.
- **El bienestar emocional** (fila combinada de ánimo, estrés o sueño) es el eje que une a **tres** cuestionarios distintos:
  - el efecto de la música en el ánimo y el sueño (Doc 1);
  - el desánimo, la irritabilidad y el sueño (Doc 2);
  - el efecto de la IA en el estrés y la carga mental (Doc 3).
- **La IA** se mide a fondo en el Documento 3 (15 preguntas) e incluye una pregunta directa sobre el estrés, lo que la conecta con el mismo eje.
- **El contexto universitario** (carrera, semestre, traslados, con quién viven) está en el Documento 2.

El tema que los datos pueden sostener es, por lo tanto, **el bienestar emocional de los estudiantes y su relación con dos hábitos cotidianos que sí están medidos: escuchar música y usar IA**.

## 4. Primeras señales en las respuestas

Antes de limpiar los datos se hizo una lectura rápida de las respuestas abiertas, contando las que contienen ciertas palabras. Son **aproximaciones** (en la Fase 3 cada respuesta se codifica con reglas revisadas), pero sirven para ver si el tema tiene relevancia en este grupo.

In [6]:
# Tabla 4. Señales en las respuestas crudas (conteo de respuestas que contienen las palabras indicadas)
senales = [
    ("Traslado de 1 hora o más", 2, columna(datos[2], "trasladarte"), r"hora"),
    ("Desánimo frecuente", 2, columna(datos[2], "decaido"),
     r"muy frecuente|casi siempre|todos los dias|\bmuch[oa]\b|bastante|^frecuente|veces a la semana"),
    ("Problemas de sueño (malo, poco, irregular)", 2, columna(datos[2], "descanso y tus horarios"),
     r"\bmal|poco|corto|irregular|pesado|no es tan|no muy|desvel"),
    ("Recurre a la IA por tiempo o carga académica", 3, columna(datos[3], "motivo principal"),
     r"tiempo|rapid|carga|muchas actividades|cansa"),
    ("Dice que la IA reduce su estrés", 3, columna(datos[3], "nivel de estres"),
     r"menos|baj|disminu|ayud|liber|quitad|aliv|control"),
    ("Escucha música a diario (más de 0 horas)", 1, columna(datos[1], "tiempo al dia"), r"[1-9]|una|dos|tres"),
    ("Escucha música en el traslado", 1, columna(datos[1], "momentos del dia").fillna(""),
     r"camino|transporte|trasporte|camion|traslad|ir y regresar|viaj"),
    ("Describe que la música mejora su ánimo", 1, columna(datos[1], "antes y despues"),
     r"mejor|sube|energ|feliz|anim|buen humor|contento|paso de"),
]
tabla_senales = pd.DataFrame([{"señal": s, "doc": d, "respuestas": int(serie.str.contains(p).sum()),
                               "de": len(serie)} for s, d, serie, p in senales])
tabla_senales["%"] = (tabla_senales["respuestas"] / tabla_senales["de"] * 100).round(0).astype(int)
tabla_senales.set_index("señal")

,doc,respuestas,de,%
señal,,,,
Traslado de 1 hora o más,2,11,19,58
Desánimo frecuente,2,10,19,53
"Problemas de sueño (malo, poco, irregular)",2,12,19,63
Recurre a la IA por tiempo o carga académica,3,12,24,50
Dice que la IA reduce su estrés,3,20,24,83
Escucha música a diario (más de 0 horas),1,23,24,96
Escucha música en el traslado,1,16,24,67
Describe que la música mejora su ánimo,1,17,24,71


**Lo que sugieren estas señales:**

- **Hay malestar en el grupo.** Más de la mitad reporta desánimo frecuente, y la mayoría menciona problemas de sueño.
- **El contexto pesa.** Más de la mitad tarda una hora o más en llegar a la universidad, y la mitad recurre a la IA por falta de tiempo o por carga académica.
- **La IA aparece ligada al estrés.** Casi todos dicen que la IA les reduce el estrés.
- **La música es cotidiana y se usa para sentirse mejor.** Casi todos escuchan música a diario, la mayoría en el traslado, y la mayoría describe que les mejora el ánimo.

Nadie había documentado cómo se relacionan estas piezas en este grupo: **qué hábitos musicales acompañan un mejor ánimo, y cómo se conecta eso con el estrés y con el uso de IA**. De ahí sale el punto de partida del proyecto (Documento de Negocio, sección 1).

## 5. Con el tema definido: ¿qué documentos se usan?

**Criterios.** Un documento se **acepta** si cumple dos condiciones:

1. Contiene variables del tema: música, estado de ánimo, estrés, sueño, concentración o uso de IA.
2. Comparte la llave *número de cuenta* con los demás.

Se **rechaza** si no aporta variables del tema, si su llave es incompatible o si solo añadiría ruido.

Como la Tabla 2 muestra que todos comparten estudiantes, la llave no descarta a ninguno y la decisión depende del primer criterio.

In [7]:
# Tabla 5. Preguntas del tema por documento (música, ánimo, estrés, sueño, concentración, IA)
del_tema = ["Música", "Ánimo y emociones", "Estrés y carga", "Sueño y descanso",
            "Concentración y desempeño", "Inteligencia artificial"]
tema_por_doc = temas.loc[del_tema].drop(columns="documentos que lo tocan").sum().rename("preguntas del tema")
nucleo = conj[1] & conj[2] & conj[3]
pd.DataFrame({"preguntas del tema": tema_por_doc,
              "comparte estudiantes con el núcleo 1∩2∩3": [len(conj[n] & nucleo) for n in DOCS]})

,preguntas del tema,comparte estudiantes con el núcleo 1∩2∩3
Doc 1,26,19
Doc 2,6,19
Doc 3,22,19
Doc 4,0,19
Doc 5,0,19
Doc 6,1,19


In [8]:
# Tabla 6. Matriz de relación y veredicto (se guarda en docs/matriz_relacion.csv)
matriz = pd.DataFrame([
    ["La música y su influencia en el estado de ánimo", "Hábitos musicales y su efecto en el ánimo",
     "Género, horas/día, momentos, plataforma, ánimo antes/después, música ante estrés, energía, concentración, sueño",
     "Sí, directa", "Sí", "ACEPTAR – principal"],
    ["Entorno Universitario y Bienestar", "Bienestar emocional y contexto",
     "Desánimo, irritabilidad, sueño, concentración, apetito, apoyo social, traslados",
     "Sí (ánimo, sueño, concentración, estrés indirecto)", "Sí", "ACEPTAR – complementario fuerte"],
    ["Desempeño académico por el uso de IA", "Uso académico de la IA",
     "Horas de IA/día, herramientas, efecto en estrés, dependencia, guía vs. atajo",
     "Sí (IA, estrés)", "Sí", "ACEPTAR – complementario medio"],
    ["Test", "Personalidad y preferencias",
     "Pasatiempo, deporte, sociabilidad, decisiones, organización del tiempo, sexo",
     "Marginal", "Sí (1 duplicado)", "RECHAZAR"],
    ["Pelis y series", "Preferencias audiovisuales",
     "Películas, series y actores favoritos/sobrevalorados", "No", "Sí", "RECHAZAR"],
    ["Relación Amor – Carrera", "Concepciones de pareja",
     "Situación sentimental, valores, señales de alerta, cita ideal", "No", "Solo vía correo", "RECHAZAR"],
], index=pd.Index(list(DOCS), name="doc"),
   columns=["Documento", "Tema central", "Variables clave",
            "¿Relación con música/ánimo/estrés/IA?", "Llave compatible", "Veredicto"])
matriz.to_csv(RAIZ / "docs" / "matriz_relacion.csv", encoding="utf-8-sig")
matriz

,Documento,Tema central,Variables clave,¿Relación con música/ánimo/estrés/IA?,Llave compatible,Veredicto
doc,,,,,,
1,La música y su influencia en el estado de ánimo,Hábitos musicales y su efecto en el ánimo,"Género, horas/día, momentos, plataforma, ánimo antes/después, música ante es...","Sí, directa",Sí,ACEPTAR – principal
2,Entorno Universitario y Bienestar,Bienestar emocional y contexto,"Desánimo, irritabilidad, sueño, concentración, apetito, apoyo social, traslados","Sí (ánimo, sueño, concentración, estrés indirecto)",Sí,ACEPTAR – complementario fuerte
3,Desempeño académico por el uso de IA,Uso académico de la IA,"Horas de IA/día, herramientas, efecto en estrés, dependencia, guía vs. atajo","Sí (IA, estrés)",Sí,ACEPTAR – complementario medio
4,Test,Personalidad y preferencias,"Pasatiempo, deporte, sociabilidad, decisiones, organización del tiempo, sexo",Marginal,Sí (1 duplicado),RECHAZAR
5,Pelis y series,Preferencias audiovisuales,"Películas, series y actores favoritos/sobrevalorados",No,Sí,RECHAZAR
6,Relación Amor – Carrera,Concepciones de pareja,"Situación sentimental, valores, señales de alerta, cita ideal",No,Solo vía correo,RECHAZAR


In [9]:
# Problemas de calidad que se atenderán en la Fase 3
print("Doc 1 – cuentas con longitud distinta de 9 dígitos:", int((llaves[1].astype(str).str.len() != 9).sum()))
print("Doc 4 – cuentas duplicadas:", int(llaves[4].duplicated().sum()))
print("Doc 6 – ¿Deseas participar?:", datos[6]["¿Deseas participar en el estudio?"].value_counts().to_dict())
print("Núcleo integrable (Doc 1 ∩ Doc 2 ∩ Doc 3):", len(nucleo), "estudiantes")

Doc 1 – cuentas con longitud distinta de 9 dígitos: 1
Doc 4 – cuentas duplicadas: 1
Doc 6 – ¿Deseas participar?: {'Sí': 20, 'No': 1}
Núcleo integrable (Doc 1 ∩ Doc 2 ∩ Doc 3): 19 estudiantes


## 6. Conclusión

- **Tema y pregunta.** Los datos sostienen un estudio sobre el **bienestar emocional** de estudiantes de 7.º semestre de MAC y su relación con **dos hábitos cotidianos medidos para las mismas personas: escuchar música y usar IA**. Las primeras señales (malestar frecuente, traslados largos, carga académica e IA ligada al estrés) justifican la problemática del Documento de Negocio.
- **Documentos.** Se usan el **1** (principal: música y ánimo), el **2** (bienestar y contexto) y el **3** (uso de IA y estrés). Se descartan el **4**, el **5** y el **6** porque sus temas no se conectan con el bienestar ni con los hábitos estudiados; comparten estudiantes, así que el rechazo es solo por tema.
- **Muestra.** El núcleo que respondió los tres documentos aceptados es de **19 estudiantes**.